# Flujo de información para generar capa regional 
#### Modelo de estimación de demanda eléctrica MERLIN_EDM
--- 
En este Notebook se empleará el modelo ``merlin_edm`` entrenado pra obtener la construcción de capas geoespaciales de demanda eléctrica en escala regional y desagregada por sector de interés. 

Los archivos de entrada para esto serán: 

- ``../data/raw/wp2_elec_input_sector_shares_raw.csv``: Los shares regionales provenientes del Balance Regional de Energía que están en formato "largo", es decir, el archivo viene como ``año | región | sector | valor`` y con datos disponibles hasta el año 2024.
- ``../data/rec_2024_2025/temperatura_regional_2024_2025.parquet``: Es la temperatura de los años 2024 y 2025 en todo Chile, en escala regional y resolución horaria. 
- ``../data/raw/reg_alias.json``: Son los alias de formato ISO de la región con respecto a su nombre disponible en las bases geoespaciales. Sirve para cruzar la información del balance regional de energía con el de las temperaturas. 

El procesamiento de estos archivos de entrada llevarán a lo siguiente: 

- Generación de rezagos temporales de temperatura (temperatura presente y 7 lags hacia atrás).
- Cálculo de series trigonométricas de hora/semana/año. 
- Condicionales de día hábil/fin de semana/feriado en escala regional. 
- Intensidades energéticas ``total_region/total_nacional`` y por sector ``total_sector_region/total_regional``.

Se debería generar una matriz de inputs que contenga los datos de todo el país para poder tomarlo como inferencia del modelo. El orden de los inputs importa para la red neuronal. Este se guarda en un archivo de texto llamado ``../data/rec_2024_2025/columns.txt`` y para cargarlo a la sesión el código es: 

```python
columns = []  # Lista vacía para que se guarden los nombres de las columnas
with open("../data/rec_2024_2025/columns.txt", "r") as f:
    for line in f: 
        columns.append(line.strip()) 

```

La matriz de inputs se usa para el modelo de red neuronal entrenado. Los outputs que se deberían obtener son: 

- ``../data/rec_2024_2025/results/capas_regionales/wp2_output_demanda_electrica_regional.gpkg``: Geocapa con los totales anuales (año 2024 y 2025) en cada región, total y por sector (RCPIT)
- ``../data/rec_2024_2025/results/capas_regionales/wp2_output_demanda_electrica_regional_ts.parquet``: Serie de tiempo con la demanda de electricidad (año 2024 y 2025) en cada región, total y por sector (RCPIT)

In [30]:
# ==========================================
# CELDA 1: CONFIGURACIÓN E IMPORTACIONES
# ==========================================
import os
import sys
import pandas as pd
import joblib
from tensorflow.keras.models import load_model
import numpy as np
import matplotlib.pyplot as plt
import holidays
import calendar
import geopandas as gpd

# 1. Configuración de Rutas Globales
BASE_DIR = os.path.abspath("..")
MODEL_PATH = os.path.join(BASE_DIR, "models", "ds_comunal", "best_merlin_mlp_global.keras")
LAYER_PATH = os.path.join(BASE_DIR, "data", "raw", "capa_regional.gpkg")
TEMP_SCALER_PATH = os.path.join(BASE_DIR, "models", "scaler_temp_global.pkl")
HIST_SHARES_PATH = os.path.join(BASE_DIR, "data", "raw", "wp2_elec_input_sector_shares_raw.csv")
TEMP_REG_PATH = os.path.join(BASE_DIR, "data", "rec_2024_2025", "temperatura_regional_2024_2025.parquet")
COLUMNS_PATH = os.path.join(BASE_DIR, "data", "rec_2024_2025", "columns.txt")
OUT_DIR = os.path.join(BASE_DIR, "data", "rec_2024_2025", "results", "capas_regionales")

os.makedirs(OUT_DIR, exist_ok=True)

# 2. Parámetros del Modelo
IS_COMUNA = 0  # Trabajamos con regiones ahora
SECTORES = ['I', 'R', 'C', 'P', 'T']
A_PARAM = np.exp(-1.1315)  
B_PARAM = 0.8988
AÑOS_TARGET = [2024, 2025]

# 3. Cargar columnas
columns = []  # Lista vacía para que se guarden los nombres de las columnas
with open("../data/rec_2024_2025/columns.txt", "r") as f:
    for line in f: 
        columns.append(line.strip()) 

# 4. Cargar modelos de red neuronal y scaler de temperatura
model = load_model(MODEL_PATH)
temp_scaler = joblib.load(TEMP_SCALER_PATH)

# 5. Cargar los shares regionales
df_hist_shares = pd.read_csv(HIST_SHARES_PATH)

# 6. Cargar los shares de temperatura
df_temp_global = pd.read_parquet(TEMP_REG_PATH)
regiones = df_temp_global["region"].unique().tolist()

In [4]:
# ==========================================
# CELDA 2: EXTRAPOLACIÓN DE SHARES Y RATIOS
# ==========================================

print("Iniciando extrapolación de consumos y cálculo de shares...")

# 1. Modificar la arquitectura del df de shares regionales
# (Nota: Verifica si tu columna se llama 'región' o 'region', aquí usaré 'region' sin tilde)
df_sec = df_hist_shares.pivot_table(
        index=['año', 'región'], 
        columns='sector', 
        values='valor', 
        aggfunc='sum'
    ).reset_index()

# Llenar posibles nulos con 0 (por si alguna región no tiene un sector específico)
df_sec = df_sec.fillna(0)

sectores = ['Industrial', 'Residencial', 'Comercial', 'Público', 'Transporte']
sectores_alias = {'Industrial': "I", 'Residencial': "R", 'Comercial': "C", 'Público': "P", 'Transporte': "T"}

# Verificación de seguridad: si un sector no existe en la tabla, lo creamos con 0
for sec in sectores:
    if sec not in df_sec.columns:
        df_sec[sec] = 0.0

# 2. Iterar por región para aislar la historia y hacer la regresión
proyecciones = []
regiones = df_sec["región"].unique().tolist()

for region in regiones:
    # Aislamos la historia exclusiva de esta región y la ordenamos cronológicamente
    df_region = df_sec[df_sec['región'] == region].sort_values('año').copy()
    proyecciones.append(df_region)
    
    years_hist = df_region['año'].values
    
    # 3. Proyectar para los años objetivo (2024, 2025)
    for target_year in AÑOS_TARGET:
        # Si el año ya viene en los datos originales, no lo sobreescribimos
        if target_year in years_hist:
            continue
            
        nueva_fila = {'año': target_year, 'región': region}
        
        for sec in sectores:
            if len(years_hist) < 2:
                # No hay historia suficiente para trazar una recta, copiamos el último año
                nueva_fila[sec] = df_region.iloc[-1][sec]
            else:
                # Extrapolación lineal exclusiva para este sector y región
                z = np.polyfit(years_hist, df_region[sec].values, 1)
                p = np.poly1d(z)
                # FILTRO FÍSICO: El consumo proyectado nunca puede ser negativo
                nueva_fila[sec] = max(0.0, p(target_year))
        
        proyecciones.append(pd.DataFrame([nueva_fila]))

# 4. Unir todo el historial + el futuro en un solo DataFrame
df_shares_proyectados = pd.concat(proyecciones, ignore_index=True)

# 5. Calcular los totales y los Shares definitivos
df_shares_proyectados['total_consumo_region'] = df_shares_proyectados[sectores].sum(axis=1)

for sec in sectores:
    alias = sectores_alias[sec] # Extrae I, R, C, P o T
    nombre_columna = f'share_{alias}'
    
    # Fracción = Sector / Total (usamos np.where para evitar división por cero)
    df_shares_proyectados[nombre_columna] = np.where(
        df_shares_proyectados['total_consumo_region'] > 0,
        df_shares_proyectados[sec] / df_shares_proyectados['total_consumo_region'],
        0.0
    )

# 6. Calcular el consumo total nacional por año
df_nacional = df_shares_proyectados.groupby('año')['total_consumo_region'].sum().reset_index()
df_nacional.rename(columns={'total_consumo_region': 'total_consumo_nacional'}, inplace=True)

# Unir el total nacional de vuelta al dataframe original
df_shares_proyectados = pd.merge(df_shares_proyectados, df_nacional, on='año', how='left')

# 7. Calcular la magnitud de la región (region_share)
df_shares_proyectados['region_comuna_share'] = df_shares_proyectados['total_consumo_region'] / df_shares_proyectados['total_consumo_nacional']

print("¡Proyección de shares completada!")

Iniciando extrapolación de consumos y cálculo de shares...
¡Proyección de shares completada!


### Cálculo de Lags de temperaturas y features de calendarios
Acá se van a calcular (para cada región): 
- Los lags de temperatura que el modelo considera para la "inercia" de $\tau = 7$ horas.
- Series de sin y cos de las horas/semanas/año del intervalo de tiempo que se va a hacer forecast. 


In [10]:
# PROBAR FUNCIONES DE time_features.py
def build_temperature_lags(
        df_temp, 
        temp_col='temperatura', 
        tau=7
):
    """
    Toma un Dataframe anual de temperatura y genera los lags, 
    usando las últimas 'tau' horas para rellenar el inicio.
    """

    df = df_temp.copy()

    for i in range(1, tau + 1): 
        # np.roll desplaza los valores. Al desplazar hacia abajo, 
        # los últimos valores pasan automáticamente al principio
        df[f'temp_t - {i}'] = np.roll(df[temp_col], i)

    return df


def build_calendar_features(df, dt_col='fecha_hora', country='CL', subdiv=None):
    """
    Construye las variables trigonométricas y categóricas del calendario
    a partir de una columna de fecha y hora.
    
    Args:
        df (pd.DataFrame): DataFrame que contiene la serie temporal.
        dt_col (str): Nombre de la columna con las fechas (Datetime).
        country (str): Código ISO del país para buscar los feriados (Defecto: 'CL').
        subdiv (str): Código  ISO de la subdivisón (región) del país (Defecto: None)
        
    Returns:
        pd.DataFrame: DataFrame con las nuevas columnas de features temporales.
    """
    df = df.copy()
    
    # 0. Asegurar que la columna de entrada sea de tipo datetime de Pandas
    if not pd.api.types.is_datetime64_any_dtype(df[dt_col]):
        df[dt_col] = pd.to_datetime(df[dt_col])
        
    # 1. Extraer componentes base
    hour = df[dt_col].dt.hour
    day_of_week = df[dt_col].dt.dayofweek  # Lunes = 0, Domingo = 6
    day_of_year = df[dt_col].dt.dayofyear
    # Detectar años bisiestos para ajustar la longitud del ciclo anual
    days_in_year = df[dt_col].dt.is_leap_year.map({True: 366, False: 365})
    
    # 2. Transformaciones Trigonométricas (Ciclos)
    # Ciclo diario (24 horas)
    df['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    
    # Ciclo semanal (7 días)
    df['dow_sin'] = np.sin(2 * np.pi * day_of_week / 7)
    df['dow_cos'] = np.cos(2 * np.pi * day_of_week / 7)
    
    # Ciclo anual (365/366 días) - *Sugerido para complementar estacionalidad
    df['doy_sin'] = np.sin(2 * np.pi * day_of_year / days_in_year)
    df['doy_cos'] = np.cos(2 * np.pi * day_of_year / days_in_year)
    
    # 3. Flags Categóricos (Fines de semana y Festivos)
    df['is_weekend'] = df[dt_col].dt.dayofweek.isin([5, 6]).astype(int)
    
    # Obtener festivos del país para los años presentes en el DataFrame
    years = df[dt_col].dt.year.unique()
    cl_holidays = holidays.country_holidays(country, subdiv=subdiv, years=years)
    
    # Mapear si la fecha cae en un día festivo
    df['is_holiday'] = df[dt_col].dt.date.apply(lambda d: d in cl_holidays).astype(int)
    
    # 4. Día laboral (Es True solo si NO es fin de semana y NO es festivo)
    df['is_working_day'] = ((df['is_weekend'] == 0) & (df['is_holiday'] == 0)).astype(int)
    
    return df


In [11]:
regiones_ISO = {
    "TARAPACÁ": "TA", 
    "ANTOFAGASTA": "AN", 
    "ATACAMA": "AT", 
    "COQUIMBO": "CO", 
    "VALPARAÍSO": "VS", 
    "LIBERTADOR GENERAL BERNARDO O'HIGGINS": "LI", 
    "MAULE": "ML", 
    "BIOBÍO": "BI", 
    "LA ARAUCANÍA": "AR", 
    "LOS LAGOS": "LL", 
    "AYSÉN DEL GENERAL CARLOS IBÁÑEZ DEL CAMPO": "AI", 
    "MAGALLANES Y DE LA ANTÁRTICA CHILENA": "MA", 
    "METROPOLITANA DE SANTIAGO": "RM", 
    "LOS RÍOS": "LR", 
    "ARICA Y PARINACOTA": "AP", 
    "ÑUBLE": "NB"
}

In [12]:
print("Generando lags de temperatura y features de calendario...")

# 1. Asegurar que la columna global sea datetime antes de entrar al bucle
df_temp_global['fecha_hora'] = pd.to_datetime(df_temp_global['fecha_hora'])
regiones_temp = df_temp_global["region"].unique().tolist()
# 2. Crear una lista vacía para ir guardando los pedazos de cada región
lista_regiones_procesadas = []

# Bucle for por regiones
for region in regiones_temp:
    print(f"Procesando características temporales para: {region}")
    region_ISO = regiones_ISO[region]
    print(f"ISO: {region_ISO}")
    
    # 3. Filtrar los datos de la región y ORDENAR cronológicamente (CRÍTICO para np.roll)
    df_reg = df_temp_global[df_temp_global["region"] == region].copy()
    df_reg = df_reg.sort_values(by="fecha_hora").reset_index(drop=True)
    
    # 4. Generar los Lags de temperatura (temp_t-1 a temp_t-7)
    df_reg = build_temperature_lags(df_reg, temp_col='temperatura', tau=7)
    
    # 5. Generar las variables de calendario (Seno, Coseno, Feriados)
    # Nota: Si tu variable 'region' contiene el código ISO exacto de la región (ej: 'RM', 'AN'), 
    # puedes pasar subdiv=region. Si contiene el nombre completo, usa subdiv=None 
    # para usar los feriados nacionales de Chile ('CL').
    df_reg = build_calendar_features(df_reg, dt_col='fecha_hora', country='CL', subdiv=region_ISO)
    
    # 6. Almacenar el DataFrame ya procesado en la lista
    lista_regiones_procesadas.append(df_reg)

# 7. Unir (Concatenar) todas las regiones procesadas en un solo DataFrame maestro
df_inputs = pd.concat(lista_regiones_procesadas, ignore_index=True)

# 8. Escalar las temperaturas (Si el modelo las espera escaladas)
# Asumiendo que temp_scaler ya fue cargado en la Celda 1
columnas_temp = ['temperatura'] + [f'temp_t - {i}' for i in range(1, 8)]
# Es importante que el scaler reciba las columnas en el mismo orden en que fue entrenado
df_inputs[columnas_temp] = temp_scaler.transform(df_inputs[columnas_temp])

print(f"¡Dataset de inputs temporales creado exitosamente! Shape: {df_inputs.shape}")

Generando lags de temperatura y features de calendario...
Procesando características temporales para: TARAPACÁ
ISO: TA
Procesando características temporales para: ANTOFAGASTA
ISO: AN
Procesando características temporales para: ATACAMA
ISO: AT
Procesando características temporales para: COQUIMBO
ISO: CO
Procesando características temporales para: VALPARAÍSO
ISO: VS
Procesando características temporales para: LIBERTADOR GENERAL BERNARDO O'HIGGINS
ISO: LI
Procesando características temporales para: MAULE
ISO: ML
Procesando características temporales para: BIOBÍO
ISO: BI
Procesando características temporales para: LA ARAUCANÍA
ISO: AR
Procesando características temporales para: LOS LAGOS
ISO: LL
Procesando características temporales para: AYSÉN DEL GENERAL CARLOS IBÁÑEZ DEL CAMPO
ISO: AI
Procesando características temporales para: MAGALLANES Y DE LA ANTÁRTICA CHILENA
ISO: MA
Procesando características temporales para: METROPOLITANA DE SANTIAGO
ISO: RM
Procesando características temporales 

In [19]:
# ==========================================
# CELDA 4: FUSIÓN GLOBAL Y MATRIZ DE INPUTS EXACTA
# ==========================================
print("Iniciando fusión global de shares proyectados y características temporales...")
df_inputs["año"] = df_inputs["fecha_hora"].dt.year
df_inputs["region_bne"] = df_inputs["region"].map(regiones_ISO)

df_shares_proyectados.rename(columns={"región": "region_bne"}, inplace=True)

# 1. Realizar el merge masivo usando 'año' y 'region' como llaves
df_master = pd.merge(df_inputs, df_shares_proyectados, on=['año', 'region_bne'], how='left')
df_master.rename(columns={'region': 'region_comuna'}, inplace=True)

# 2. Agregar el flag espacial exigido por la arquitectura de la red neuronal global
# IS_COMUNA viene definido como 0 desde la Celda 1
df_master['is_comuna'] = IS_COMUNA

# 3. Control de Calidad: Verificar si hubo pérdidas en el cruce de datos
filas_nulas = df_master['share_R'].isna().sum()
if filas_nulas > 0:
    print(f"⚠️ ¡Advertencia! Hay {filas_nulas} registros horarias que no encontraron su share anual.")
    print("Esto suele pasar si los nombres de las regiones difieren entre las bases (ej. tildes o alias).")
    # Opcional: llenar con 0 o con el promedio si deseas forzar la ejecución
    # df_master = df_master.fillna(0) 
else:
    print("✅ Cruce de datos completado exitosamente. Cero registros nulos encontrados.")

# 4. Validación estructural del orden del Vector de Entrada (Matriz X)
print("\nValidando alineación con la lista exigida por la Red Neuronal (columns.txt)...")

# Buscamos si hay alguna columna en columns.txt que no exista en nuestro DataFrame fusionado
columnas_faltantes = [col for col in columns if col not in df_master.columns]

if columnas_faltantes:
    print(f"❌ ERROR CRÍTICO: El modelo exige columnas que no están en el DataFrame: {columnas_faltantes}")
    print("👉 Consejo: Revisa si hay discrepancias de nombres (ej. 'hour_sin' vs 'sin_hora' o 'temp_t - 1' vs 'temp_t-1').")
    print("Modifica el nombre en el DataFrame para que calce exactamente con lo que pide columns.txt")
else:
    print(f"✅ Alineación perfecta. Las {len(columns)} variables requeridas están presentes.")
    
    # 5. Extracción de la Matriz X Definitiva (Tensor Ciego para Keras)
    # Al pasar la lista 'columns', forzamos a que las columnas queden en el ORDEN EXACTO en que se entrenó la red
    X_inferencia = df_master[columns].values
    
    print(f"\n⚡ ¡Matriz X construida con éxito!")
    print(f"-> Dimensiones finales del input para model.predict(): {X_inferencia.shape}")


Iniciando fusión global de shares proyectados y características temporales...
✅ Cruce de datos completado exitosamente. Cero registros nulos encontrados.

Validando alineación con la lista exigida por la Red Neuronal (columns.txt)...
✅ Alineación perfecta. Las 24 variables requeridas están presentes.

⚡ ¡Matriz X construida con éxito!
-> Dimensiones finales del input para model.predict(): (280704, 24)


### Agregar parámetros de estandarización y desagregación

In [21]:
def _get_hours_in_year(year):
    """
    Determina la cantidad exacta de horas en un año (bisiesto o normal).
    """
    # Se emplea el método .isleap(year) que entrega un booleano si es bisiesto o no
    is_leap = calendar.isleap(year)
    return 8784 if is_leap else 8760


def calculate_scaling_parameters(total_anual_mwh, consumos_anuales_sectores, year, a, b):
    """
    Calcula mu y sigma global de la zona, así como los valores 'sin sector' 
    para la etapa de desagregación de la demanda.
    
    Args:
        total_anual_mwh (float): Consumo físico total proyectado para el año completo (MWh).
        consumos_anuales_sectores (dict): Diccionario con el consumo anual de cada sector.
                                          Ej: {'I': 1500.5, 'R': 2000.0, 'C': 500.0, ...}
        year (int): Año a evaluar (para definir 8760 vs 8784 horas).
        a (float): Parámetro 'a' de la curva de correlación empírica (sigma = a * mu^b).
        b (float): Parámetro 'b' de la curva de correlación empírica.
        
    Returns:
        dict: Diccionario que contiene mu_total, sigma_total, mu_sin_X, sigma_sin_X.
    """
    horas_anio = _get_hours_in_year(year)
    
    # 1. Parámetros Globales (Total de la Zona)
    mu_total = total_anual_mwh / horas_anio
    sigma_total = a * (mu_total ** b)
    
    resultados = {
        'mu_total': mu_total,
        'sigma_total': sigma_total
    }
    
    # 2. Parámetros "Sin Sector" para los Escenarios Puros (Desagregación)
    for sector, consumo_sector in consumos_anuales_sectores.items():
        # Restamos el consumo del sector al total. Aplicamos max(0.0) como filtro 
        # físico por si la matemática de proyección arrojara inconsistencias ínfimas.
        total_sin_sector = max(0.0, total_anual_mwh - consumo_sector)
        
        mu_sin = total_sin_sector / horas_anio
        
        # Omitimos cálculo de sigma si mu es 0 (ej. zona sin consumo) para evitar errores matemáticos
        if mu_sin > 0:
            sigma_sin = a * (mu_sin ** b)
        else:
            sigma_sin = 0.0
            
        resultados[f'mu_sin_{sector}'] = mu_sin
        resultados[f'sigma_sin_{sector}'] = sigma_sin
        
    return resultados

In [25]:
# ==========================================
# CELDA 5: CÁLCULO DE MU Y SIGMA (MACRO) Y MERGE A DF_MASTER
# ==========================================

print("Calculando parámetros de escalamiento (mu, sigma) por región y año...")

# 1. Extraer una tabla única (macro) para no recalcular 8760 veces lo mismo
# Asumiendo que tu df_master (o df_shares_proyectados) tiene una columna con la demanda total anual 
# Si se llama distinto, cambia 'demanda_total_anual_mwh' por el nombre correcto.
cols_macro = ['año', 'region_bne', 'total_consumo_region'] + [f'share_{s}' for s in SECTORES]
df_macro = df_shares_proyectados[cols_macro].drop_duplicates().reset_index(drop=True)

# 2. Función envoltorio para aplicar el cálculo fila por fila a la matriz macro
def apply_scaling(row):
    year = int(row['año'])
    total_mwh = row['total_consumo_region'] * 1000  # GWh a MWh
    
    # Reconstruir los consumos anuales absolutos (MWh) a partir de los shares
    consumos_sectores = {s: total_mwh * row[f'share_{s}'] for s in SECTORES}
    
    # Llamamos a la función
    return calculate_scaling_parameters(
        total_anual_mwh=total_mwh,
        consumos_anuales_sectores=consumos_sectores,
        year=year,
        a=A_PARAM,
        b=B_PARAM
    )

# 3. Aplicar y expandir los diccionarios resultantes en nuevas columnas
df_params = df_macro.apply(apply_scaling, axis=1, result_type='expand')

# Concatenamos los parámetros calculados a nuestra tabla macro
df_macro_completa = pd.concat([df_macro[['año', 'region_bne']], df_params], axis=1)

# 4. Unir (Merge) estos parámetros de vuelta a la base horaria (df_master)
df_master = pd.merge(df_master, df_macro_completa, on=['año', 'region_bne'], how='left')

print(f"✅ ¡Parámetros agregados con éxito! Dimensiones de df_master: {df_master.shape}")

Calculando parámetros de escalamiento (mu, sigma) por región y año...
✅ ¡Parámetros agregados con éxito! Dimensiones de df_master: (280704, 47)


### Inferencia 

In [26]:
def predict_and_disaggregate(model, df_inputs, df_metadata, feature_cols, sectores=['I', 'R', 'C', 'P', 'T']):
    """
    Realiza la estimación de demanda eléctrica total y la desagregación sectorial 
    mediante el método de resta de escenarios.
    
    Args:
        model (keras.Model): Modelo MLP global pre-entrenado.
        df_inputs (pd.DataFrame): DataFrame solo con los features (X) numéricos y escalados.
        df_metadata (pd.DataFrame): DataFrame con metadatos asociados a las filas de df_inputs 
                                    (debe contener mu_total, sigma_total, mu_sin_X, sigma_sin_X).
        feature_cols (list): Lista con el orden exacto de las columnas que traga el modelo.
        sectores (list): Lista de los identificadores de los sectores.
        
    Returns:
        pd.DataFrame: df_metadata enriquecido con las curvas de demanda total y por sector en MWh.
    """
    df_res = df_metadata.copy()
    
    # Aseguramos el orden estricto de las columnas para la red neuronal
    X_total = df_inputs[feature_cols].values

    # ---------------------------------------------------------
    # PASO 1: PREDICCIÓN TOTAL
    # ---------------------------------------------------------
    print("-> Calculando demanda total...")
    y_pred_scaled = model.predict(X_total, batch_size=2048).flatten()
    
    # Desescalamiento con parámetros globales
    df_res['demanda_total_pred'] = (y_pred_scaled * df_res['sigma_total']) + df_res['mu_total']
    df_res['demanda_total_pred'] = df_res['demanda_total_pred'].clip(lower=0.0) # Filtro físico

    # ---------------------------------------------------------
    # PASO 2: DESAGREGACIÓN SECTORIAL (MÉTODO DE RESTA)
    # ---------------------------------------------------------
    for sector in sectores:
        print(f"-> Desagregando sector: {sector}")
        df_sim = df_inputs.copy()
        
        # 1. Apagar el sector objetivo (llevar su share a cero)
        col_share = f'share_{sector}'
        df_sim[col_share] = 0.0

        # 2. Predecir el escenario "Sin el Sector"
        X_sim = df_sim[feature_cols].values
        y_pred_sin_scaled = model.predict(X_sim, batch_size=2048).flatten()

        # 3. Desescalar utilizando mu y sigma SIN el sector
        col_mu_sin = f'mu_sin_{sector}'
        col_sigma_sin = f'sigma_sin_{sector}'
        
        demanda_sin_pred = (y_pred_sin_scaled * df_res[col_sigma_sin]) + df_res[col_mu_sin]
        demanda_sin_pred = demanda_sin_pred.clip(lower=0.0)

        # 4. Obtener demanda del sector por diferencia (Total - Predicción sin el sector)
        col_demanda_sector = f'demanda_pred_{sector}'
        df_res[col_demanda_sector] = df_res['demanda_total_pred'] - demanda_sin_pred
        df_res[col_demanda_sector] = df_res[col_demanda_sector].clip(lower=0.0)

    return df_res

In [29]:
# ==========================================
# CELDA 6: INFERENCIA Y MÉTODO KUSUMOTO (ESCENARIOS PUROS)
# ==========================================
print("Iniciando motor de desagregación sectorial (Red Neuronal)...")

# 1. Separar los Inputs estrictos (Matriz X Ciega)
# La lista 'columns' viene del txt cargado en la Celda 1. Esto garantiza orden perfecto.
df_inputs = df_master[columns].copy()

# 2. Separar la Metadata (Lo que usaremos para desescalar y trazar resultados)
cols_metadata = ['fecha_hora', 'año', 'region_comuna', 'region_bne'] + \
                ['mu_total', 'sigma_total'] + \
                [f'mu_sin_{s}' for s in SECTORES] + \
                [f'sigma_sin_{s}' for s in SECTORES]

df_metadata = df_master[cols_metadata].copy()

# 3. Ejecutar la función de inferencia (modelo cargado previamente)
df_resultados = predict_and_disaggregate(
    model=model, 
    df_inputs=df_inputs, 
    df_metadata=df_metadata, 
    feature_cols=columns,     # Le pasamos la lista para que X_sim no se desordene 
    sectores=SECTORES
)

print("\n⚡ ¡Inferencia y Desagregación Completadas!")
print("Muestra de los resultados (Total y Sectores en MWh):")
print(df_resultados[['fecha_hora', 'region_comuna', 'demanda_total_pred'] + [f'demanda_pred_{s}' for s in SECTORES]].head())

Iniciando motor de desagregación sectorial (Red Neuronal)...
-> Calculando demanda total...
138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 449us/step
-> Desagregando sector: I
138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 432us/step
-> Desagregando sector: R
138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 421us/step
-> Desagregando sector: C
138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 395us/step
-> Desagregando sector: P
138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 396us/step
-> Desagregando sector: T
138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 405us/step

⚡ ¡Inferencia y Desagregación Completadas!
Muestra de los resultados (Total y Sectores en MWh):
           fecha_hora region_comuna  demanda_total_pred  demanda_pred_I  \
0 2024-01-01 00:00:00      TARAPACÁ          311.657605      252.577176   
1 2024-01-01 01:00:00      TARAPACÁ          294.038043      238.760446   
2 2024-01-01 02:00:00      TARAPACÁ          281.540157      229.594147   
3 2024-01-01 03:00:00      TARAPACÁ          271.712053      222.444581   
4 2024-01-01 04:00:00      TARAPACÁ          263

In [34]:
import pandas as pd
import geopandas as gpd
import os

# --- 0. CONFIGURACIÓN INICIAL ---
SECTORES = ['R', 'C', 'P', 'I', 'T']
# Rutas de salida (ajusta según tu estructura real)
OUTPUT_PARQUET = os.path.join(OUT_DIR, "wp2_output_demanda_electrica_regional_ts.parquet")
OUTPUT_GPKG = os.path.join(OUT_DIR, "wp2_output_demanda_electrica_regional.gpkg")
RUTA_CAPA_BASE = LAYER_PATH  # Tu mapa base con las geometrías de las regiones

# --- 1. AJUSTES AL DATAFRAME DE RESULTADOS ---
# Renombrar de vuelta 'region_comuna' a 'region' y asegurar el nombre de la fecha a 'timestamp'
df_resultados = df_resultados.rename(columns={
    'region_comuna': 'region',
    'fecha_hora': 'timestamp' # Si ya se llamaba timestamp, esto no afecta
})

# Asegurarse de que el timestamp sea un objeto datetime de pandas
df_resultados['timestamp'] = pd.to_datetime(df_resultados['timestamp'])

# --- 2. SERIE DE TIEMPO (.parquet) ---
print("Generando serie de tiempo horaria...")

# Definir las columnas exactas que solicitaste para la serie de tiempo
cols_sectores = [f"demanda_pred_{sector}" for sector in SECTORES]
cols_ts = ["timestamp", "region", "demanda_total_pred"] + cols_sectores

# Filtrar el DataFrame
df_time_series = df_resultados[cols_ts].copy()
cols_sectores_mwh = [f"demanda_{sector}_MWh" for sector in SECTORES]
rename_cols = {f"demanda_pred_{sector}": f"demanda_{sector}_mwh" for sector in SECTORES}
rename_cols["demanda_total_pred"] = "demanda_MWh"
df_time_series = df_time_series.rename(columns=rename_cols)

# Guardar en parquet para el equipo de ingesta
os.makedirs(os.path.dirname(OUTPUT_PARQUET), exist_ok=True)
df_time_series.to_parquet(OUTPUT_PARQUET, engine='pyarrow', index=False)
print(f"-> Serie de tiempo guardada en: {OUTPUT_PARQUET}")


# --- 3. AGREGACIÓN ANUAL Y GEOCAPA (.gpkg) ---
print("\nCalculando totales anuales y generando geocapa...")

# Extraer el año para la agrupación
df_resultados['año'] = df_resultados['timestamp'].dt.year

# Agrupar por año y región, sumando los MWh de todo el año
df_anual_mwh = df_resultados.groupby(['año', 'region'])[["demanda_total_pred"] + cols_sectores].sum().reset_index()

# Conversión de unidades: MWh a GWh (1 GWh = 1,000 MWh)
df_anual_mwh['demanda_total_GWh'] = df_anual_mwh['demanda_total_pred'] / 1000.0

cols_sectores_gwh = []
for sector in SECTORES:
    col_mwh = f"demanda_pred_{sector}"
    col_gwh = f"demanda_{sector}_GWh"
    df_anual_mwh[col_gwh] = df_anual_mwh[col_mwh] / 1000.0
    cols_sectores_gwh.append(col_gwh)

# Seleccionar solo las columnas objetivo para la geocapa
cols_geocapa = ["año", "region", "demanda_total_GWh"] + cols_sectores_gwh
df_datos_geocapa = df_anual_mwh[cols_geocapa].copy()

# Cargar la capa vectorial (GeoDataFrame) de regiones
# IMPORTANTE: Asegúrate de que esta capa tenga una columna llamada 'region' con los mismos nombres
gdf_regiones = gpd.read_file(RUTA_CAPA_BASE)

# Unir los datos anuales con las geometrías (Merge)
gdf_final = gdf_regiones.merge(df_datos_geocapa, on="region", how="inner")

# Guardar como GeoPackage
os.makedirs(os.path.dirname(OUTPUT_GPKG), exist_ok=True)
gdf_final.to_file(OUTPUT_GPKG, driver="GPKG")
print(f"-> Geocapa anual guardada exitosamente en: {OUTPUT_GPKG}")

Generando serie de tiempo horaria...
-> Serie de tiempo guardada en: /home/ica/MERLIN_EDM/prototipo_3/data/rec_2024_2025/results/capas_regionales/wp2_output_demanda_electrica_regional_ts.parquet

Calculando totales anuales y generando geocapa...
-> Geocapa anual guardada exitosamente en: /home/ica/MERLIN_EDM/prototipo_3/data/rec_2024_2025/results/capas_regionales/wp2_output_demanda_electrica_regional.gpkg
